# 05 — Feature engineering

Tạo đặc trưng từ kết luận của EDA (notebook 04). Mỗi nhóm dưới đây gắn với một kết quả đã đo được,
hoặc ghi rõ là **giả thuyết** nếu EDA chưa kiểm tra.

| Căn cứ | Feature tạo ra | Mục EDA |
|---|---|---|
| Chu kỳ ngày, tuần, mùa rõ | `hour_sin/cos`, `dow_sin/cos`, `month_sin/cos` | VI |
| Tốc độ gió chi phối, lặng gió làm bụi tích tụ | `wind_calm`, `wind_roll6` | 10.3, 10.7 |
| Hướng gió khác biệt nhỏ, giữ để bước chọn quyết định | `wind_sin/cos` | 10.6 |
| PM10 gần như đồng biến với PM2.5 (Spearman 0.95) | PM10 trực tiếp, `PM10_PM25_ratio` | IX |
| *Giả thuyết:* áp suất tăng nhanh báo hiệu thời tiết ổn định | `Pressure_anom_24` | kiểm tra ở 10.4b |
| Chỉ mưa lớn mới rửa được bụi | `hours_since_rain`, `rain_24h_sum`, `Accumulated Hours of Rain` | 10.5 |
| *Giả thuyết:* ngày lễ giảm giao thông | `IsHoliday` | kiểm tra ở 6.3 |
| Độ ẩm đất, lớp biên, cháy có tín hiệu sau khi loại mùa | 5 cột dữ liệu ngoài | 13.1 |
| PM2.5 phụ thuộc lịch sử của chính nó | 32 feature lag / rolling (chỉ cho mô hình dạng bảng) | VII |

**Không dùng:** `AQI` (chỉ số tổng hợp), `Dew Point` (trùng nhiệt độ r = 0.86 và áp suất r = -0.87, mất tương quan sau khi loại mùa, mục 13.2).

**Hai nguyên tắc của bản này**

1. **Giữ nguyên lưới giờ liên tục**, không `dropna()` toàn bảng. Notebook 06 cắt cửa sổ theo vị trí dòng,
   nên nếu xóa dòng giữa chuỗi thì một cửa sổ có thể ghép hai đoạn thời gian xa nhau và target "+24h" thành một giờ khác.
   Giờ nào thiếu thì để NaN, bước sau tự loại mẫu hỏng.
2. **Không dùng thông tin tương lai.** Lấp chỗ trống chỉ dùng giá trị quá khứ (`ffill`),
   ngưỡng cắt ngoại lai chỉ tính trên phần Train.

**Bước thời gian:** 1 giờ.
**Đầu vào:** `data/processed/cleaned_data_v2.csv`
**Đầu ra:** `feature_data.csv`, `split_dates.csv`, `feature_groups.json`

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT / 'data' / 'processed'

pd.set_option('display.width', 160)

## I. Đọc dữ liệu và kiểm tra lưới giờ

In [ ]:
df = pd.read_csv(DATA_DIR / 'cleaned_data_v2.csv', parse_dates=['Local Time'])
df = df.set_index('Local Time').sort_index()

assert not df.index.duplicated().any(), 'Có timestamp trùng'

# Ép về lưới giờ liên tục: giờ nào không có dòng sẽ hiện ra dưới dạng NaN thay vì biến mất
n_before = len(df)
df = df.asfreq('h')
print('Số giờ thêm vào để lấp lưới:', len(df) - n_before)

# Bỏ metadata, AQI và các cột EDA kết luận không dùng
drop_cols = [c for c in ['City', 'Country Code', 'Timezone', 'AQI', 'Dew Point',
                         'Fire Dist Min', 'wd_sin', 'wd_cos'] if c in df.columns]
df = df.drop(columns=drop_cols)
print('Đã bỏ:', drop_cols)

print('Kích thước:', df.shape)
print('Từ', df.index.min(), 'đến', df.index.max())
display(df.head())

## II. Tạo cột target và cố định mốc chia dữ liệu

`PM25_target` là giá trị dùng để chấm điểm. `PM25_was_missing` đánh dấu giờ gốc không có quan trắc.

`PM25_suspect` đánh dấu giờ có **PM2.5 ≥ PM10**, điều không thể xảy ra về vật lý (mục 4.2 của EDA).
Cột này chỉ là cờ, không đưa vào mô hình và không sửa target. Có thể dùng sau để loại các giờ này khỏi bước chấm điểm.

Mốc chia 70/15/15 theo **thời gian**, ghi ra `split_dates.csv` để các notebook sau đọc lại.

In [ ]:
df['PM25_target'] = df['PM25'].copy()
df['PM25_was_missing'] = df['PM25_target'].isna()
df['PM25_suspect'] = (df['PM25'] >= df['PM10']) & df['PM25'].notna() & df['PM10'].notna()

n = len(df)
train_end = df.index[int(n * 0.70) - 1]
val_start = df.index[int(n * 0.70)]
val_end = df.index[int(n * 0.85) - 1]
test_start = df.index[int(n * 0.85)]

split_table = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Start': [df.index[0], val_start, test_start],
    'End': [train_end, val_end, df.index[-1]]
})
display(split_table)
split_table.to_csv(DATA_DIR / 'split_dates.csv', index=False)

is_train = df.index <= train_end

print('Số giờ target thiếu:', int(df['PM25_was_missing'].sum()))
print('Số giờ PM2.5 >= PM10:', int(df['PM25_suspect'].sum()))
print('Đã lưu split_dates.csv')

## III. Đặc trưng thời gian

PM2.5 có chu kỳ ngày, tuần và mùa. Mỗi chu kỳ dùng cả sin và cos để không mất thông tin vị trí trong chu kỳ
(nếu chỉ dùng sin thì 6 giờ sáng và 6 giờ chiều có cùng giá trị).

In [ ]:
df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)

df['dow_sin'] = np.sin(2 * np.pi * df.index.dayofweek / 7)
df['dow_cos'] = np.cos(2 * np.pi * df.index.dayofweek / 7)

df['month_sin'] = np.sin(2 * np.pi * (df.index.month - 1) / 12)
df['month_cos'] = np.cos(2 * np.pi * (df.index.month - 1) / 12)

display(df[['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']].head())

## IV. Gió

EDA mục 10.7: ở cả 8 hướng, PM2.5 đều giảm khi gió mạnh (hướng Bắc: 44.0 khi gió yếu, 26.0 khi gió mạnh),
nên **tốc độ gió chi phối mạnh hơn hướng gió**. Mục 10.3 cho thấy PM2.5 gần như không đổi ở hai nhóm gió thấp nhất
(39.0 ở ≤ 1.0 m/s, 38.7 ở 1.0–1.6 m/s) rồi mới giảm dần, xuống 30.0 ở nhóm gió mạnh nhất.
Vì vậy ngưỡng lặng gió đặt ở 1.6 m/s, đúng chỗ đường cong bắt đầu đổi.

Hướng gió là góc 0–360 nên cần sin/cos. Khác biệt theo hướng nhỏ (median 32.4–35.0), giữ lại để bước chọn quyết định.

In [ ]:
WIND_CALM_THRESHOLD = 1.6  # m/s, ranh giới nhóm 2 và 3 ở EDA mục 10.3

wind_rad = np.deg2rad(df['Wind Direction'])
df['wind_sin'] = np.sin(wind_rad)
df['wind_cos'] = np.cos(wind_rad)

df['wind_calm'] = (df['Wind Speed'] <= WIND_CALM_THRESHOLD).astype(float)
df.loc[df['Wind Speed'].isna(), 'wind_calm'] = np.nan

# Gió trung bình 6 giờ trước, không gồm giờ hiện tại
df['wind_roll6'] = df['Wind Speed'].shift(1).rolling(6).mean()

print(f"Tỷ lệ giờ lặng gió: {df['wind_calm'].mean() * 100:.1f}%")
display(df[['Wind Direction', 'wind_sin', 'wind_cos', 'wind_calm', 'wind_roll6']].head(8))

## V. Tỷ lệ bụi thô và bụi mịn

PM10 gồm cả hạt lớn, PM2.5 chỉ hạt mịn. Tỷ số cao thường là bụi đường hoặc công trường,
tỷ số thấp thiên về khí thải động cơ và khói.

Tỷ số này **bùng nổ khi PM2.5 rất nhỏ** (ví dụ PM10 = 20, PM2.5 = 0.5 cho tỷ số 40).
Cách xử lý: mẫu số tối thiểu 1 µg/m³, rồi cắt ở phân vị 1 và 99 **tính trên Train**
để ngưỡng không chứa thông tin của Validation và Test.

In [ ]:
ratio = df['PM10'] / df['PM25'].clip(lower=1.0)

ratio_low, ratio_high = ratio[is_train].quantile([0.01, 0.99])
df['PM10_PM25_ratio'] = ratio.clip(ratio_low, ratio_high)

print(f'Ngưỡng cắt (từ Train): {ratio_low:.3f} đến {ratio_high:.3f}')
print('Số giờ bị cắt:', int(((ratio < ratio_low) | (ratio > ratio_high)).sum()))
display(pd.DataFrame({'Trước cắt': ratio.describe(), 'Sau cắt': df['PM10_PM25_ratio'].describe()}).round(3))

## VI. Áp suất đổi nhanh (giả thuyết)

EDA mục 10.4: tương quan áp suất và PM2.5 (0.36) **gần như biến mất** sau khi loại mùa (0.013),
nên giá trị áp suất tuyệt đối chủ yếu mang thông tin mùa. Phần có thể còn lại là **áp suất đang thay đổi**:
áp suất tăng nhanh so với 24 giờ trước thường đi kèm khối không khí lạnh, ổn định.

Đây là giả thuyết, EDA mục 10.4b kiểm tra nó. Nếu tương quan gần 0 thì bước chọn đặc trưng sẽ tự loại.
Dùng `shift(1)` để không lẫn giá trị hiện tại.

In [ ]:
pressure_prev = df['Pressure'].shift(1)
df['Pressure_anom_24'] = pressure_prev - pressure_prev.rolling(24).mean()

display(df[['Pressure', 'Pressure_anom_24']].describe().round(3))

## VII. Mưa và hiệu ứng rửa bụi

EDA mục 10.5: mưa nhỏ hầu như không rửa được bụi, chỉ mưa từ khoảng 2.5 mm trở lên mới rõ
(không mưa 36.4, mưa nhẹ 29.3, mưa từ 2.5 mm 22.3 µg/m³).
Vì vậy ngoài lượng mưa tức thời cần thêm số giờ mưa liên tục, tổng mưa 24 giờ và số giờ đã tạnh.

- `Accumulated Hours of Rain`: đang mưa liên tục được bao nhiêu giờ (0 khi không mưa).
- `hours_since_rain`: đã tạnh bao nhiêu giờ (giờ khô đầu tiên sau mưa = 1, đang mưa = 0).
- `rain_24h_sum`: tổng lượng mưa 24 giờ gần nhất, tính cả giờ hiện tại.

Đoạn khô ở đầu chuỗi chưa biết trận mưa trước đó nên `hours_since_rain` để NaN cho tới trận mưa đầu tiên.

In [ ]:
precip = df['Precipitation'].fillna(0)
raining = (precip > 0).astype(int)
groups = (raining != raining.shift()).cumsum()

df['Accumulated Hours of Rain'] = raining.groupby(groups).cumsum().astype(float)

# Đếm từ 1 trong đoạn khô, đang mưa thì bằng 0
dry_count = raining.groupby(groups).cumcount() + 1
df['hours_since_rain'] = dry_count.where(raining == 0, 0).astype(float)

# Đoạn khô đầu chuỗi chưa có trận mưa nào trước đó
first_rain = raining.idxmax() if raining.any() else df.index[-1]
df.loc[df.index < first_rain, 'hours_since_rain'] = np.nan

df['rain_24h_sum'] = precip.rolling(24).sum()

display(df[['Precipitation', 'Accumulated Hours of Rain', 'hours_since_rain', 'rain_24h_sum']]
        .loc['2025-09-01':'2025-09-03'].head(30))

## VIII. Ngày lễ Việt Nam (giả thuyết)

Giả định: ngày lễ làm lưu lượng xe giảm, nhất là Tết Nguyên Đán kéo dài cả tuần.
EDA mục 6.3 kiểm tra giả định này bằng cách so ngày lễ với ngày thường cùng tháng.
Lưu ý Tết còn có đốt vàng mã và pháo hoa nên hiệu ứng có thể ngược chiều.

Danh sách ngày nghỉ chính thức 2022–2026 (khai báo thủ công vì API không trả về).

In [ ]:
VN_HOLIDAYS = {
    # Tết Dương lịch
    '2022-01-01', '2023-01-01', '2024-01-01', '2025-01-01', '2026-01-01',
    # Giỗ Tổ Hùng Vương
    '2022-04-10', '2023-04-29', '2024-04-18', '2025-04-07', '2026-04-26',
    # 30/4 - 1/5
    '2022-04-30', '2022-05-01', '2023-04-30', '2023-05-01',
    '2024-04-30', '2024-05-01', '2025-04-30', '2025-05-01',
    '2026-04-30', '2026-05-01',
    # Quốc khánh 2/9 (2022 nghỉ 1-4/9)
    '2022-09-01', '2022-09-02', '2022-09-03', '2022-09-04',
    '2023-09-01', '2023-09-02', '2023-09-03',
    '2024-08-31', '2024-09-01', '2024-09-02', '2024-09-03',
    '2025-08-30', '2025-08-31', '2025-09-01', '2025-09-02',
    '2026-08-31', '2026-09-01', '2026-09-02',
    # Tết Nguyên Đán
    '2022-01-29', '2022-01-30', '2022-01-31', '2022-02-01', '2022-02-02',
    '2022-02-03', '2022-02-04', '2022-02-05', '2022-02-06',
    '2023-01-20', '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24',
    '2023-01-25', '2023-01-26',
    '2024-02-08', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12',
    '2024-02-13', '2024-02-14',
    '2025-01-25', '2025-01-26', '2025-01-27', '2025-01-28', '2025-01-29',
    '2025-01-30', '2025-01-31', '2025-02-01', '2025-02-02',
    '2026-02-14', '2026-02-15', '2026-02-16', '2026-02-17', '2026-02-18',
    '2026-02-19', '2026-02-20', '2026-02-21', '2026-02-22',
}

df['IsHoliday'] = df.index.strftime('%Y-%m-%d').isin(VN_HOLIDAYS).astype(int)

print('Số giờ rơi vào ngày lễ:', int(df['IsHoliday'].sum()))
print('Số ngày lễ có trong dữ liệu:', df.loc[df['IsHoliday'] == 1].index.normalize().nunique())

## IX. Lấp chỗ trống ngắn của dữ liệu ngoài

`Boundary Layer Height` thiếu khoảng 10.7% vì ERA5-Land không phủ hết khoảng thời gian.

Cách xử lý: **lấp bằng giá trị giờ trước (`ffill`), tối đa 3 giờ liên tiếp**.
Bản trước dùng nội suy tuyến tính hai chiều, tức là dùng cả giá trị *sau* chỗ trống để lấp,
nên tại thời điểm dự báo mô hình đã "thấy" một phần tương lai. `ffill` chỉ dùng quá khứ.

Khoảng trống dài hơn 3 giờ vẫn để NaN. Dòng vẫn được giữ trong bảng, nên lưới giờ không bị đứt.

In [ ]:
external_numeric = [c for c in ['Soil Moisture', 'Boundary Layer Height', 'Wind Speed 10m',
                                 'Fire Count 100km 3d', 'Fire FRP 100km 7d'] if c in df.columns]

before = df[external_numeric].isna().sum()
df[external_numeric] = df[external_numeric].ffill(limit=3)
after = df[external_numeric].isna().sum()

print(pd.DataFrame({'Trước': before, 'Sau': after, 'Đã lấp': before - after}))

## X. Feature lịch sử PM2.5 (chỉ cho mô hình dạng bảng)

EDA cho thấy PM2.5 phụ thuộc mạnh vào lịch sử của chính nó. Tuy nhiên:

- **GRU / LSTM ở notebook 06 không dùng nhóm này**, vì chúng tự đọc lịch sử qua cửa sổ thời gian.
  Đưa thêm lag vào cửa sổ là lặp lại thông tin mô hình đã thấy.
- Nhóm này giữ lại cho **mô hình dạng bảng** (XGBoost, Random Forest, hồi quy tuyến tính),
  vốn chỉ nhìn một dòng nên cần lịch sử được "trải" thành cột.
- EDA mục 7.1 cho thấy một phần tương quan ở lag dài là do mùa vụ, nên các lag 96–168 giờ có thể ít giá trị.

Lag chia ba tầng: ngắn (1–12 giờ), trung (24–48 giờ, cùng giờ hôm qua, hôm kia), dài (72–168 giờ).
Rolling dùng `shift(1)` để không lẫn giá trị hiện tại.

In [ ]:
LAGS = [1, 2, 3, 6, 8, 12, 24, 36, 48, 72, 96, 120, 144, 168]

history = {}
for lag in LAGS:
    history[f'PM25_lag_{lag}'] = df['PM25'].shift(lag)

shifted = df['PM25'].shift(1)
for w in (6, 12, 24, 72):
    history[f'PM25_roll_mean_{w}'] = shifted.rolling(w).mean()
    history[f'PM25_roll_std_{w}'] = shifted.rolling(w).std()
for w in (6, 24):
    history[f'PM25_roll_max_{w}'] = shifted.rolling(w).max()
    history[f'PM25_roll_min_{w}'] = shifted.rolling(w).min()

history['PM25_diff_1'] = shifted.diff(1)
history['PM25_diff_24'] = shifted.diff(24)

history = pd.DataFrame(history, index=df.index)

same_3d = [f'PM25_lag_{d}' for d in (24, 48, 72)]
same_7d = [f'PM25_lag_{d}' for d in (24, 48, 72, 96, 120, 144, 168)]
history['PM25_same_hour_mean_3d'] = history[same_3d].mean(axis=1, skipna=False)
history['PM25_same_hour_mean_7d'] = history[same_7d].mean(axis=1, skipna=False)
history['PM25_same_hour_std_7d'] = history[same_7d].std(axis=1, skipna=False)
history['PM25_same_hour_max_7d'] = history[same_7d].max(axis=1, skipna=False)

# Ghép một lần để tránh cảnh báo DataFrame bị phân mảnh
df = pd.concat([df, history], axis=1)
pm25_history = list(history.columns)

print('Số feature lịch sử PM2.5:', len(pm25_history))
display(df[pm25_history].tail(3).round(2))

## XI. Danh sách đặc trưng theo nhóm

Danh sách được ghi ra `feature_groups.json` để notebook 06 đọc lại, thay vì chép tay tên cột ở hai nơi
(bản trước 06 đòi `Solar Radiation` nhưng 05 không xuất cột này).

In [ ]:
feature_groups = {
    'target_history': ['PM25'],
    'pollutants': ['PM10', 'CO', 'NO2', 'SO2', 'O3'],
    'weather_base': ['Temperature', 'Relative Humidity', 'Precipitation', 'Clouds',
                     'Pressure', 'Wind Speed', 'UV Index', 'Solar Radiation'],
    'weather_derived': ['wind_sin', 'wind_cos', 'wind_calm', 'wind_roll6', 'Pressure_anom_24'],
    'external': external_numeric,
    'derived': ['PM10_PM25_ratio', 'hours_since_rain', 'rain_24h_sum',
                'Accumulated Hours of Rain', 'IsHoliday'],
    'time': ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos'],
    'pm25_history_tabular_only': pm25_history,
}

# Cột khí tượng nào không có trong file gốc thì bỏ khỏi nhóm và báo ra
for group, cols in feature_groups.items():
    absent = [c for c in cols if c not in df.columns]
    if absent:
        print(f'Nhóm {group}: không có cột {absent}, bỏ khỏi danh sách')
        feature_groups[group] = [c for c in cols if c in df.columns]

input_columns = [c for cols in feature_groups.values() for c in cols]
assert len(input_columns) == len(set(input_columns)), 'Có cột bị trùng giữa các nhóm'

for group, cols in feature_groups.items():
    print(f'  {group:<28} {len(cols):>2} cột')
print('Tổng feature:', len(input_columns))
print('  Dùng cho GRU/LSTM (không gồm lịch sử PM2.5):', len(input_columns) - len(pm25_history))

## XII. Lưu và kiểm tra

Bảng giữ **toàn bộ lưới giờ**, kể cả các dòng còn NaN. Ba kiểm tra trước khi lưu:

1. Index là chuỗi giờ liên tục, không thiếu, không trùng.
2. Không feature nào dùng giá trị tương lai: dịch PM2.5 tương lai thì các feature không được đổi.
3. Báo số giờ đầy đủ mọi feature, tách theo có và không có nhóm lịch sử PM2.5.

In [ ]:
keep_cols = input_columns + ['PM25_target', 'PM25_was_missing', 'PM25_suspect']
data_features = df[keep_cols].copy()

# 1. Lưới giờ liên tục
step = data_features.index.to_series().diff().dropna()
assert (step == pd.Timedelta(hours=1)).all(), 'Lưới giờ bị đứt'
assert data_features.index.is_unique

# 2. Kiểm tra rò rỉ: làm nhiễu PM2.5 từ một mốc trở đi, mọi feature trước mốc phải giữ nguyên
probe = df.index[len(df) // 2]
leak_test = df.copy()
leak_test.loc[leak_test.index >= probe, 'PM25'] += 1000
shifted_probe = leak_test['PM25'].shift(1)
recomputed = {
    'PM25_lag_1': leak_test['PM25'].shift(1),
    'PM25_roll_mean_24': shifted_probe.rolling(24).mean(),
    'PM25_diff_1': shifted_probe.diff(1),
}
for col, series in recomputed.items():
    same = np.allclose(series.loc[:probe], df[col].loc[:probe], equal_nan=True)
    assert same, f'{col} dùng giá trị tương lai'
print('Kiểm tra rò rỉ: đạt')

# 3. Số giờ đầy đủ
seq_cols = [c for c in input_columns if c not in pm25_history]
full_seq = data_features[seq_cols].notna().all(axis=1)
full_all = data_features[input_columns].notna().all(axis=1)
print(f'Tổng số giờ: {len(data_features)}')
print(f'Giờ đủ feature cho GRU/LSTM : {int(full_seq.sum())} ({full_seq.mean() * 100:.1f}%)')
print(f'Giờ đủ cả lịch sử PM2.5     : {int(full_all.sum())} ({full_all.mean() * 100:.1f}%)')

na_left = data_features[seq_cols].isna().sum()
print('Cột còn NaN (không tính lịch sử PM2.5):')
print(na_left[na_left > 0].to_string() if (na_left > 0).any() else '  không có')

data_features = data_features.reset_index()
data_features.to_csv(DATA_DIR / 'feature_data.csv', index=False, encoding='utf-8-sig')

with open(DATA_DIR / 'feature_groups.json', 'w', encoding='utf-8') as f:
    json.dump(feature_groups, f, ensure_ascii=False, indent=2)

print('Đã lưu feature_data.csv:', data_features.shape)
print('Đã lưu feature_groups.json')

## XIII. Việc cần làm ở notebook 06

Vì 05 không còn xóa dòng, notebook 06 cần chỉnh ba chỗ để khớp:

1. Đọc nhóm feature từ `feature_groups.json` thay cho danh sách chép tay ở mục IV:
   ```python
   groups = json.load(open(DATA_DIR / 'feature_groups.json', encoding='utf-8'))
   pollutants, weather_base = groups['pollutants'], groups['weather_base']
   weather_derived, external = groups['weather_derived'], groups['external']
   derived, time_features = groups['derived'], groups['time']
   ```
2. Đặt `CHECK_WINDOW = max(WINDOWS)` để mọi cửa sổ, kể cả 96 giờ, đều không chứa NaN.
   Bản cũ chỉ kiểm tra 48 giờ, nên cửa sổ 96 giờ có thể lọt NaN vào GRU.
3. Không cần sửa gì ở phần cắt cửa sổ: lưới giờ giờ đã liên tục nên "24 dòng sau" đúng bằng "24 giờ sau".